In [ ]:
# Imports

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

In [2]:
# Load CSV Data

train = pd.read_csv("data_train.csv", index_col=0)
test = pd.read_csv("smiles_test.csv", index_col=0)
sample_sub = pd.read_csv("sample_submission.csv", index_col=0)

task_cols = [c for c in train.columns if c.startswith("task")]

In [3]:
# Create Fingerprints

def smiles_to_morgan(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(n_bits, dtype=np.uint8)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.uint8)
    Chem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

X_train = np.vstack(train["smiles"].apply(smiles_to_morgan).values)
X_test = np.vstack(test["smiles"].apply(smiles_to_morgan).values)


[14:55:48] WARNING: not removing hydrogen atom without neighbors
[14:55:48] WARNING: not removing hydrogen atom without neighbors
[14:55:49] WARNING: not removing hydrogen atom without neighbors
[14:55:49] WARNING: not removing hydrogen atom without neighbors
[14:55:50] WARNING: not removing hydrogen atom without neighbors
[14:55:50] WARNING: not removing hydrogen atom without neighbors
[14:55:50] WARNING: not removing hydrogen atom without neighbors
[14:55:50] WARNING: not removing hydrogen atom without neighbors


In [4]:
# 3) CV + Training per task

oof_preds = pd.DataFrame(index=train.index, columns=task_cols, dtype=float)
test_preds = pd.DataFrame(index=test.index, columns=task_cols, dtype=float)

for task in task_cols:
    y_raw = train[task].values
    
    # bekannte Labels: -1 oder +1
    mask = y_raw != 0
    X_task = X_train[mask]
    y_task = y_raw[mask]
    
    # Mapping: -1 -> 0, +1 -> 1
    y_task = (y_task == 1).astype(int)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    test_fold_preds = []

    for tr_idx, va_idx in skf.split(X_task, y_task):
        X_tr, X_va = X_task[tr_idx], X_task[va_idx]
        y_tr, y_va = y_task[tr_idx], y_task[va_idx]

        clf = LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
        clf.fit(X_tr, y_tr)

        va_pred = clf.predict_proba(X_va)[:, 1]
        test_pred = clf.predict_proba(X_test)[:, 1]

        original_idx = train.index[mask][va_idx]
        oof_preds.loc[original_idx, task] = va_pred
        test_fold_preds.append(test_pred)

    test_preds[task] = np.mean(test_fold_preds, axis=0)

In [ ]:
# 4) local AUC analog to challenge script

auc_scores = []
for task in task_cols:
    y_true = train[task].values
    mask = y_true != 0
    y_bin = (y_true[mask] == 1).astype(int)
    y_score = oof_preds.loc[train.index[mask], task].values
    auc = roc_auc_score(y_bin, y_score)
    auc_scores.append(auc)
    print(task, auc)

print("Mean AUC:", np.mean(auc_scores))

task1 0.8233650856693827
task2 0.8046622042341219
task3 0.5815771042144668
task4 0.6024473684210526
task5 0.8211147086031452
task6 0.887185452693474
task7 0.9197436062202802
task8 0.7891797834406339
task9 0.633672488383272
task10 0.7035278045644487
task11 0.7058733803691871
Mean AUC: 0.7520317260739514


In [6]:
# Save submission 
submission = test_preds.copy()
submission.to_csv("submission.csv")